# Tavily Search → menu retrieval → OpenAI extraction

Tavily basic Search discovers sources. Python fetches actual restaurant/menu pages and follows daily-menu and PDF links. OpenAI then extracts structured meals from that fetched text, with no OpenAI web search. Python verifies quoted dates, day headings and price evidence before saving the existing app JSON. No Tavily Research or Research output schema is used.

## 1. Load API keys and helpers

Keys stay in `.env`. `OPENAI_MODEL` selects the extraction model (default `gpt-4.1-mini`). Setup reloads the Python modules without making paid requests; rerun it after changing helper code. The workflow functions now live in `menu_pipeline.py`.


In [ ]:
import os
import sys
import importlib
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import load_dotenv
from openai import OpenAI

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import tavily_agent
import menu_retrieval
import menu_pipeline as pipeline
importlib.reload(tavily_agent)
importlib.reload(menu_retrieval)
importlib.reload(pipeline)
from tavily_agent import save_menus, recheck_saved_run

load_dotenv(project_root / ".env", override=True)
tavily_key = os.getenv("TAVILY_API_KEY")
extraction_model = os.getenv("OPENAI_MODEL") or "gpt-4.1-mini"
client = OpenAI()
diagnostics_path = project_root / "data/diagnostics/latest-run.json"
print("Tavily key loaded:", bool(tavily_key))
print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Extraction model:", extraction_model)


## 2. Restaurants and date

The notebook and terminal command share `restaurants.json`. Edit names, addresses and stable menu pages there; changing weekly PDF links are discovered automatically. Set `selected_restaurants = restaurants[:1]` to test only Spilka. Saving replaces the file with the selected restaurants, so use the full list for the full app.


In [ ]:
restaurants = pipeline.load_restaurants(project_root / "restaurants.json")
selected_restaurants = restaurants  # Use restaurants[:1] to test only Spilka.

today = datetime.now(ZoneInfo("Europe/Bratislava")).date()
print("Requested date:", today)


## 3. Discover with Tavily basic Search

One basic-search credit per selected restaurant. This cell does not call OpenAI. Search snippets are not treated as menu contents; the next step fetches the sources. Query dates do not establish menu validity.

In [ ]:
search_results = pipeline.search_restaurants(selected_restaurants, today, tavily_key)
for result in search_results:
    print(result["restaurant"]["name"], len(result["sources"]), "discovered sources")
    print("Usage:", result["usage"])


## 4. Retrieve actual menu content

No AI request or Tavily credits here. Python fetches the stable menu page plus relevant search results, follows likely menu links up to two levels, and reads PDF text. It limits each restaurant to eight downloads, 8 MB per download, and 45,000 text characters per source. Blocked pages, JavaScript-only content and image/scanned menus are reported; there is no vision fallback yet. Inspect the URLs and text before extraction. This step does not use stale search snippets as substitute evidence.

In [ ]:
retrieved_results = pipeline.retrieve_restaurants(search_results, selected_restaurants, today)
for result in retrieved_results:
    print(result["restaurant"]["name"])
    for source in result["sources"]:
        print(source["kind"], source["url"], len(source["content"]), "characters")
    for error in result["errors"]:
        print("Retrieval issue:", error)


## 5. Extract and validate

OpenAI reads the fetched text without a web-search tool. Python checks the evidence. Matching verified results are reused by default; failed results are retried. Model requests cost tokens; no automatic retries are made.

To intentionally compare another model on every restaurant, change `retry_failed_only` to `False`. Sources and raw responses are checkpointed before local validation. Inspect `menu_results` and the original menus before saving: evidence checks do not prove completeness.


In [ ]:
menu_results = pipeline.extract_menus(
    retrieved_results, selected_restaurants, today,
    client=client,
    model=extraction_model,
    diagnostics_path=diagnostics_path,
    previous_results=globals().get("menu_results", []),
    retry_failed_only=True,
)
for result in menu_results:
    print(result["restaurant"]["name"], result["status"], len(result["meals"]), "meals")
    print(result.get("reason") or result["notes"])


## 6. Save verified meals

No API requests. All selected restaurants must be verified for the requested date; otherwise the previous `menus.json` stays untouched. Review before saving. The existing flat meals/soups/labels format is unchanged. Refresh the app after saving.

In [ ]:
menus_data = save_menus(menu_results, selected_restaurants, today, project_root / "menus.json")
print("Saved", len(menus_data["meals"]), "meals")


## Optional: recheck saved responses

After changing validation code, run setup to reload helpers, then this cell. No API requests and no menus.json write. It updates `menu_results` in memory for review and saving.


In [ ]:
menu_results = recheck_saved_run(diagnostics_path)
for result in menu_results:
    print(result["restaurant"]["name"], result["status"], len(result["meals"]), "meals")
    print(result.get("reason") or result["notes"])


## Optional: recover after local processing fails

If a paid response returned but local processing failed, reload helpers in setup and run this cell instead of repeating extraction. It reads the checkpoint from disk, so a kernel restart does not lose the paid response. It makes no API requests and does not write menus.json. Resume the extraction cell afterward if other restaurants remain unprocessed.


In [ ]:
menu_results = pipeline.recover_extraction(diagnostics_path)
for result in menu_results:
    print(result["restaurant"]["name"], result["status"], len(result["meals"]), "meals")
    print(result.get("reason") or result["notes"])
